In [1]:
import sys, os
sys.path.append(os.path.join('/home/module'))

from Bq_Connect import BQDataProcessor  
sql = BQDataProcessor(env="dev") 

/opt/conda/lib/python3.8/site-packages/google/cloud/bigquery_storage_v1/__init__.py:57: FutureWarning: You are using a non-supported Python version (3.8.16).  Google will not post any further updates to google.cloud.bigquery_storage_v1 supporting this Python version. Please upgrade to the latest Python version, or at least to Python 3.9, and then update google.cloud.bigquery_storage_v1.
  warnings.warn(


[BQDataProcessor] Connected → env=DEV, project=pgd-dev-data-analytics, SA=default (/home/jupyterhub/SA/jupyter-pgd-dev-data-analytics-296ca984dd2f.json)


In [6]:
# DEST_TABLE       = "datamart_audit.dm_det_tbl_trx_te_new" 
DEST_TABLE = "pgd-dev-data-analytics.datamart_audit.dm_det_tbl_trx_te" # target yang benar yang ini yang '_new' cuman buat nyoba doang waktu itu
SELECT_QUERY = """
SELECT 
    t.cif, 
    t.norek, 
    t.branch_code, 
    cast (t.create_date as timestamp) create_date,
    t.product_code, 
    t.jenis_transaksi, 
    t.hutang_transaksi AS nominal_trx, 
    t.norek_kredit,
    t.no_ref_id,
    det.berat
FROM pgd-prod-data-analytics.datalake.pgd_tbl_transaksi_tabungan t
LEFT JOIN pgd-prod-data-analytics.datalake.pgd_tbl_e_transaksi_emas_det det 
    ON t.id = det.id
WHERE 
    t.jenis_transaksi IN ('BUY', 'SALE', 'TRANSFER', 'TRANSFERKS')
    AND t.create_date >= DATETIME(
        DATE_TRUNC(
            DATE_SUB(CURRENT_DATE(), INTERVAL 3 MONTH),
            MONTH
        )
    )

"""


In [7]:
sql.execute_ddl(f"""create or replace table {DEST_TABLE} as 
{SELECT_QUERY}
""")

print(sql.read(f"select * from {DEST_TABLE} limit 10"))
print(sql.read(f"select count(*) from {DEST_TABLE} limit 10") )

[DDL] Executing: create or replace table pgd-dev-data-analytics.datamart_audit.dm_det_tbl_trx_te as 

SELECT 
    t.cif, 
    t.norek, 
 ...
[DDL] Success.
[READ] Executing query...
[READ] Success → 10 row(s) returned.
          cif             norek branch_code               create_date  \
0  1004333172  1363825620000884       13638 2026-02-02 17:34:59+00:00   
1  1009520493  1028719620003500       10287 2026-02-04 18:44:59+00:00   
2  9002513615  1287225620001920       12872 2026-04-06 20:22:40+00:00   
3  6008998782  6067025620001812       60670 2026-01-05 07:18:29+00:00   
4  1036322104  1173526629000657       11735 2026-03-26 14:57:57+00:00   
5  9003836214  1221019620000381       12210 2026-04-09 09:52:53+00:00   
6  1031206292  1326626629006518       13266 2026-03-11 08:55:12+00:00   
7  1002945297  1127224620000903       11272 2026-02-06 11:02:16+00:00   
8  1035731041  1368225629040145       13682 2026-01-29 14:07:04+00:00   
9  1026268368  1330226620000406       13302 2026-04